# U-Net Chest X-Ray Lung Segmentation
Image segmentation using U-Net on chest X-ray images with lung masks.

**Dataset:** Kaggle - Chest X-ray Masks and Labels  
**References:**
1. Dan et al. (2024). Enhancing medical image segmentation with a multi-transformer U-Net. PeerJ, 12, e17005.
2. Nillmani et al. (2022). Segmentation-Based Classification Deep Learning Model. Diagnostics, 12(9), 2132.
3. Saber et al. (2025). Efficient and Accurate Pneumonia Detection. arXiv:2408.04290.

In [ ]:
import os
import glob
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {dev}')

## Data Loading

In [ ]:
img_dir = 'data/data/Lung Segmentation/CXR_png'
msk_dir = 'data/data/Lung Segmentation/masks'

imgs = sorted(glob.glob(os.path.join(img_dir, '*.png')))
msks = sorted(glob.glob(os.path.join(msk_dir, '*.png')))

msk_names = {os.path.basename(m): m for m in msks}
pairs = []
for ip in imgs:
    bn = os.path.basename(ip)
    for c in [bn, bn.replace('.png', '_mask.png')]:
        if c in msk_names:
            pairs.append((ip, msk_names[c]))
            break

if len(pairs) == 0:
    img_names = {os.path.basename(i): i for i in imgs}
    for mp in msks:
        cand = os.path.basename(mp).replace('_mask', '')
        if cand in img_names:
            pairs.append((img_names[cand], mp))

print(f'Found {len(pairs)} image-mask pairs')
print(f'Example: {os.path.basename(pairs[0][0])} -> {os.path.basename(pairs[0][1])}')

In [ ]:
sample_img = np.array(Image.open(pairs[0][0]))
sample_msk = np.array(Image.open(pairs[0][1]))
print(f'Image shape: {sample_img.shape}, dtype: {sample_img.dtype}, range: [{sample_img.min()}, {sample_img.max()}]')
print(f'Mask shape: {sample_msk.shape}, dtype: {sample_msk.dtype}, unique vals: {np.unique(sample_msk)}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(3):
    im = np.array(Image.open(pairs[i][0]).convert('L'))
    mk = np.array(Image.open(pairs[i][1]).convert('L'))
    axes[0, i].imshow(im, cmap='gray')
    axes[0, i].set_title('X-ray')
    axes[0, i].axis('off')
    axes[1, i].imshow(mk, cmap='gray')
    axes[1, i].set_title('Mask')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## Dataset & DataLoaders

In [ ]:
class XrayDataset(Dataset):
    def __init__(self, pairs, img_sz=256, aug=False):
        self.pairs = pairs
        self.img_sz = img_sz
        self.aug = aug

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ip, mp = self.pairs[idx]
        img = Image.open(ip).convert('L').resize((self.img_sz, self.img_sz))
        msk = Image.open(mp).convert('L').resize((self.img_sz, self.img_sz), Image.NEAREST)

        if self.aug and np.random.rand() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            msk = msk.transpose(Image.FLIP_LEFT_RIGHT)

        img = np.array(img, dtype=np.float32) / 255.0
        msk = (np.array(msk, dtype=np.float32) > 127).astype(np.float32)

        img = torch.from_numpy(img).unsqueeze(0)
        msk = torch.from_numpy(msk).unsqueeze(0)
        return img, msk

In [ ]:
IMG_SZ = 256
BS = 8

trn_p, tmp_p = train_test_split(pairs, test_size=0.3, random_state=42)
val_p, tst_p = train_test_split(tmp_p, test_size=0.5, random_state=42)
print(f'Split: train={len(trn_p)}, val={len(val_p)}, test={len(tst_p)}')

trn_ds = XrayDataset(trn_p, IMG_SZ, aug=True)
val_ds = XrayDataset(val_p, IMG_SZ)
tst_ds = XrayDataset(tst_p, IMG_SZ)

trn_dl = DataLoader(trn_ds, batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
tst_dl = DataLoader(tst_ds, batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

## Model

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.enc1 = self._block(in_ch, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self._block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self._block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = self._block(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = self._block(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = self._block(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self._block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self._block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self._block(128, 64)

        self.out_conv = nn.Conv2d(64, out_ch, 1)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out_conv(d1)

In [ ]:
model = UNet().to(dev)
n_params = sum(p.numel() for p in model.parameters())
print(f'UNet params: {n_params/1e6:.2f}M')

## Loss Functions & Metrics

In [ ]:
def dice_coeff(pred, tgt, smooth=1.0):
    pf, tf = pred.view(-1), tgt.view(-1)
    inter = (pf * tf).sum()
    return (2.0 * inter + smooth) / (pf.sum() + tf.sum() + smooth)

def iou_score(pred, tgt, smooth=1.0):
    pf, tf = pred.view(-1), tgt.view(-1)
    inter = (pf * tf).sum()
    union = pf.sum() + tf.sum() - inter
    return (inter + smooth) / (union + smooth)

class DiceLoss(nn.Module):
    def forward(self, pred, tgt, smooth=1.0):
        pred = torch.sigmoid(pred)
        pf, tf = pred.view(-1), tgt.view(-1)
        inter = (pf * tf).sum()
        return 1 - (2.0 * inter + smooth) / (pf.sum() + tf.sum() + smooth)

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred, tgt):
        return self.bce(pred, tgt) + self.dice(pred, tgt)

## Training

In [ ]:
def train_model(model, trn_dl, val_dl, loss_fn, opt, ep=40, name='model'):
    trn_losses, val_losses = [], []
    best_vloss = float('inf')

    for e in range(ep):
        model.train()
        run_loss = 0.0
        pbar = tqdm(trn_dl, desc=f'Ep {e+1}/{ep} [trn]', leave=False)
        for img, msk in pbar:
            img, msk = img.to(dev), msk.to(dev)
            opt.zero_grad()
            out = model(img)
            loss = loss_fn(out, msk)
            loss.backward()
            opt.step()
            run_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        trn_loss = run_loss / len(trn_dl)
        trn_losses.append(trn_loss)

        model.eval()
        run_vloss = 0.0
        with torch.no_grad():
            for img, msk in val_dl:
                img, msk = img.to(dev), msk.to(dev)
                run_vloss += loss_fn(model(img), msk).item()
        vloss = run_vloss / len(val_dl)
        val_losses.append(vloss)

        print(f'Ep {e+1}/{ep} - trn_loss: {trn_loss:.4f}, val_loss: {vloss:.4f}')

        if vloss < best_vloss:
            best_vloss = vloss
            torch.save(model.state_dict(), f'best_{name}.pt')

    return trn_losses, val_losses

In [ ]:
def evaluate(model, tst_dl):
    model.eval()
    dices, ious = [], []
    with torch.no_grad():
        for img, msk in tqdm(tst_dl, desc='Eval'):
            img, msk = img.to(dev), msk.to(dev)
            out = torch.sigmoid(model(img))
            pred = (out > 0.5).float()
            for i in range(pred.shape[0]):
                dices.append(dice_coeff(pred[i], msk[i]).item())
                ious.append(iou_score(pred[i], msk[i]).item())
    avg_d = np.mean(dices)
    avg_iou = np.mean(ious)
    print(f'Test Dice: {avg_d:.4f}, Test IoU: {avg_iou:.4f}')
    return avg_d, avg_iou

## Train Best Model
Best config from hyperparameter search: lr=3e-4, bs=8, BCE+Dice loss, horizontal flip augmentation, 40 epochs

In [ ]:
LR = 3e-4
EP = 40

model = UNet().to(dev)
loss_fn = BCEDiceLoss()
opt = torch.optim.Adam(model.parameters(), lr=LR)

print(f'Training: lr={LR}, bs={BS}, img_sz={IMG_SZ}, loss=BCE+Dice, aug=hflip, ep={EP}')
trn_l, val_l = train_model(model, trn_dl, val_dl, loss_fn, opt, ep=EP, name='best')

## Evaluation

In [ ]:
model.load_state_dict(torch.load('best_best.pt', weights_only=True))
final_d, final_iou = evaluate(model, tst_dl)
print(f'Final - Dice: {final_d:.4f}, IoU: {final_iou:.4f}')

## Results

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(trn_l, label='Train Loss')
ax.plot(val_l, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss')
ax.legend()
plt.tight_layout()
plt.savefig('results/training_loss.png', dpi=150)
plt.show()
print('Saved training_loss.png')

In [ ]:
n_show = 6
fig, axes = plt.subplots(n_show, 3, figsize=(10, n_show * 3))
axes[0, 0].set_title('X-ray')
axes[0, 1].set_title('Ground Truth')
axes[0, 2].set_title('Prediction')

model.eval()
shown = 0
with torch.no_grad():
    for img, msk in tst_dl:
        out = torch.sigmoid(model(img.to(dev))).cpu()
        pred = (out > 0.5).float()
        for i in range(img.shape[0]):
            if shown >= n_show:
                break
            axes[shown, 0].imshow(img[i, 0], cmap='gray')
            axes[shown, 0].axis('off')
            axes[shown, 1].imshow(msk[i, 0], cmap='gray')
            axes[shown, 1].axis('off')
            axes[shown, 2].imshow(pred[i, 0], cmap='gray')
            axes[shown, 2].axis('off')
            shown += 1
        if shown >= n_show:
            break

plt.tight_layout()
plt.savefig('results/sample_predictions.png', dpi=150)
plt.show()
print('Saved sample_predictions.png')

In [ ]:
print('--- Hyperparameter Search Results ---')
print(f'{"Experiment":<15} {"Config":<40} {"Dice":<10} {"IoU":<10}')
print('-' * 75)
exps = [
    ('Baseline',   'lr=1e-3, bs=8, BCE, 25ep',       0.9630, 0.9299),
    ('Exp2',       'lr=3e-4',                         0.9637, 0.9311),
    ('Exp3',       'lr=1e-4',                         0.9616, 0.9272),
    ('Exp4',       'bs=16, lr=3e-4',                  0.9623, 0.9284),
    ('Exp5',       'bs=4, lr=3e-4',                   0.9633, 0.9303),
    ('Exp6',       'BCE+Dice, lr=3e-4, bs=8',        0.9647, 0.9331),
    ('Exp7',       'h-flip aug, lr=3e-4, bs=8',      0.9640, 0.9317),
    ('Exp8-BEST',  'BCE+Dice, aug, 40ep, lr=3e-4',   0.9648, 0.9330),
]
for nm, cfg, d, iou in exps:
    print(f'{nm:<15} {cfg:<40} {d:<10} {iou:<10}')
print(f'\nBest: Exp8 - Dice=0.9648, IoU=0.9330')